# Notebook 04 - TargetDiff Generation on EGFR Wild-Type (1M17)

**Goals:**
1. Set up a working conda environment for TargetDiff (openbabel, lmdb, PyTorch 2.4.0+cu121, PyG stack)
2. Patch the sampling config to point at our checkpoint
3. Smoke test (20 mols), then full batch generation (10 x 100 mols, resumable)
4. Merge per-molecule SDFs into `results/generated/1M17_raw.sdf`

**Runtime:** GPU required. Pro users: recommend **L4** (~2x faster than T4, 24 GB VRAM, cheap compute units).
On L4, 1000 molecules takes ~5 h; on T4 ~10 h; on A100 ~2.5 h.

**Why so many setup cells:** TargetDiff pins old deps. Colab ships PyTorch 2.10+cu128 by default,
but `torch-scatter` / `torch-sparse` have no prebuilt wheels for that. We therefore downgrade to
**torch 2.4.0 + cu121** inside a conda env (so `openbabel` also installs cleanly).

**Kernel restart warning:** `condacolab.install()` in Cell 2 RESTARTS the kernel. After it restarts,
continue from Cell 3. Do NOT re-run Cells 1-2.


In [ ]:
# -- Cell 1: Mount Drive + clone repo + set paths (run FIRST) -------------------
REPO_URL = "https://github.com/Jonathan-Ye-1/egfr-drug-design-eecs6895-final-project.git"

import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = '/content/drive/MyDrive/egfr_diffusion'
if not os.path.exists(os.path.join(PROJECT_ROOT, '.git')):
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_ROOT], check=True)
else:
    subprocess.run(['git', '-C', PROJECT_ROOT, 'pull'], check=False)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

TARGETDIFF = os.path.join(PROJECT_ROOT, 'external', 'targetdiff')
print('PROJECT_ROOT =', PROJECT_ROOT)
print('TARGETDIFF   =', TARGETDIFF)

import torch
print('\nPyTorch (current):', torch.__version__, '| CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> pick L4 / A100 / T4 GPU'
print('GPU              :', torch.cuda.get_device_name(0))


In [ ]:
# -- Cell 2: Install condacolab. THIS RESTARTS THE KERNEL. --
# After the restart, skip Cells 1-2 and start from Cell 3.
!pip install -q condacolab
import condacolab
condacolab.install()


In [ ]:
# -- Cell 3: Post-restart environment setup --
# On T4 runtimes condacolab successfully switches the kernel to Python 3.11.
# On L4 runtimes it often does NOT — the kernel stays on system Python 3.12
# while conda lives at /usr/local/lib/python3.11/. That's fine: we install
# TargetDiff's deps into the 3.11 env and later run ALL sampling via
# subprocess using /usr/local/bin/python3.11, so the notebook kernel's
# version is irrelevant.

import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = '/content/drive/MyDrive/egfr_diffusion'
TARGETDIFF   = os.path.join(PROJECT_ROOT, 'external', 'targetdiff')
CONDA_PY     = '/usr/local/bin/python3.11'     # conda env we target for sampling
for p in (PROJECT_ROOT, TARGETDIFF):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(PROJECT_ROOT)

print('kernel python :', sys.executable, '|', sys.version.split()[0])
print('CONDA_PY      :', CONDA_PY, '|', 'exists:', os.path.exists(CONDA_PY))
print('PROJECT_ROOT  :', PROJECT_ROOT)
print('TARGETDIFF    :', TARGETDIFF)

# conda-provided: openbabel + lmdb (pip-built openbabel hangs)
!conda install -y -c conda-forge openbabel python-lmdb 2>&1 | tail -5

# Install torch 2.4.0 + cu121 + PyG stack into the 3.11 env. We use CONDA_PY's
# pip explicitly so nothing leaks into the 3.12 system env.
pip = [CONDA_PY, '-m', 'pip', 'install', '--no-cache-dir']
subprocess.run(pip + ['--index-url', 'https://download.pytorch.org/whl/cu121',
                      'torch==2.4.0', 'torchvision==0.19.0', 'torchaudio==2.4.0'],
               check=False)
subprocess.run(pip + ['--no-deps', '-f', 'https://data.pyg.org/whl/torch-2.4.0+cu121.html',
                      'torch-scatter', 'torch-sparse', 'torch-cluster', 'torch-spline-conv'],
               check=False)
subprocess.run(pip + ['torch-geometric', 'easydict', 'omegaconf', 'pyyaml', 'rdkit', 'tqdm', 'meeko', 'scipy', 'numpy', 'pandas', 'networkx', 'scikit-learn'], check=False)

# Verify inside the 3.11 env (not the kernel python!)
probe = '''
import sys, torch, torch_scatter, torch_sparse, torch_cluster, torch_geometric
import lmdb, openbabel, easydict, omegaconf, yaml, rdkit, tqdm, scipy
print("python     :", sys.version.split()[0])
print("torch      :", torch.__version__, "| cuda:", torch.version.cuda)
print("gpu        :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("scatter    :", torch_scatter.__version__)
print("sparse     :", torch_sparse.__version__)
print("cluster    :", torch_cluster.__version__)
print("geometric  :", torch_geometric.__version__)
print("lmdb+babel : ok"); print("rdkit     :", rdkit.__version__)
print("READY")
'''
r = subprocess.run([CONDA_PY, '-c', probe], capture_output=True, text=True)
print('\n--- 3.11 env probe ---')
print(r.stdout if r.returncode == 0 else r.stderr[-1500:])


In [ ]:
# -- Cell 4: Patch the official sampling.yml to point at our checkpoint --
import os, yaml

with open(os.path.join(TARGETDIFF, 'configs', 'sampling.yml')) as f:
    sampling_cfg = yaml.safe_load(f)

sampling_cfg['model']['checkpoint'] = os.path.join(
    TARGETDIFF, 'checkpoints', 'pretrained_diffusion.pt'
)

custom_cfg = os.path.join(PROJECT_ROOT, 'configs', 'targetdiff_sampling.yml')
os.makedirs(os.path.dirname(custom_cfg), exist_ok=True)
with open(custom_cfg, 'w') as f:
    yaml.dump(sampling_cfg, f, sort_keys=False)

print('Patched config:', custom_cfg)
print(open(custom_cfg).read())


## Smoke test - 20 molecules

Quick sanity check before committing to the full 1000. On T4 ~12 min; on L4 ~6 min; on A100 ~3 min.


In [ ]:
# -- Cell 5: Smoke test (20 molecules, batch=10) --
# Subprocess uses conda Python 3.11 so torch 2.4.0 + PyG stack resolve correctly.
import subprocess, os, time

CONDA_PY = '/usr/local/bin/python3.11'
POCKET  = f'{PROJECT_ROOT}/data/pockets/1M17_pocket.pdb'
CONFIG  = f'{PROJECT_ROOT}/configs/targetdiff_sampling.yml'
OUT_DIR = f'{PROJECT_ROOT}/results/generated/1M17_smoke'
SCRIPT  = os.path.join(TARGETDIFF, 'scripts', 'sample_for_pocket.py')
os.makedirs(OUT_DIR, exist_ok=True)

wrapper = (
    f"import sys, runpy; sys.path.insert(0, {TARGETDIFF!r}); "
    f"sys.argv = ['sample_for_pocket.py', {CONFIG!r}, "
    f"'--pdb_path', {POCKET!r}, "
    f"'--num_samples', '20', '--batch_size', '10', "
    f"'--result_path', {OUT_DIR!r}]; "
    f"runpy.run_path({SCRIPT!r}, run_name='__main__')"
)
env = os.environ.copy()
env['PYTHONPATH'] = TARGETDIFF + os.pathsep + env.get('PYTHONPATH', '')

print('Launching smoke test (20 mols) via', CONDA_PY, '...', flush=True)
t0 = time.time()
r = subprocess.run([CONDA_PY, '-c', wrapper],
                   cwd=TARGETDIFF, capture_output=True, text=True, env=env)
print(f'Elapsed: {time.time()-t0:.1f}s | rc={r.returncode}')
if r.returncode != 0:
    print('STDERR:\n', r.stderr[-2500:])

print('\n=== Output ===')
for root, _, files in os.walk(OUT_DIR):
    for f in files:
        p = os.path.join(root, f)
        print(f'  {p}  ({os.path.getsize(p)/1024:.1f} KB)')


## Full batch generation - 10 x 100 mols, resumable

Each batch writes to its own `batch_XX/` folder. Re-running this cell after a
disconnect skips any batch whose `sample.pt` already exists.

On **L4 with batch_size=20** the run should finish in ~5 h. On A100 you can push
`PER_GPU_BATCH` to 40 for another speedup.


In [ ]:
# -- Cell 6: Batch generation (10 x 100 mols, resumable, uses conda Python 3.11) --
import subprocess, os, time

CONDA_PY = '/usr/local/bin/python3.11'
CONFIG   = f'{PROJECT_ROOT}/configs/targetdiff_sampling.yml'
POCKET   = f'{PROJECT_ROOT}/data/pockets/1M17_pocket.pdb'
BATCHES  = f'{PROJECT_ROOT}/results/generated/1M17_batches'
SCRIPT   = os.path.join(TARGETDIFF, 'scripts', 'sample_for_pocket.py')

N_BATCHES      = 10
MOLS_PER_BATCH = 100
PER_GPU_BATCH  = 20     # T4: 10. L4 (24GB): 20 safe, 40 usually ok. A100: 40.

os.makedirs(BATCHES, exist_ok=True)

def batch_done(path):
    return os.path.exists(os.path.join(path, 'sample.pt'))

t_all = time.time()
for i in range(N_BATCHES):
    out_dir = os.path.join(BATCHES, f'batch_{i:02d}')
    if batch_done(out_dir):
        print(f'[batch {i:02d}] already done, skipping')
        continue
    os.makedirs(out_dir, exist_ok=True)
    print(f'\n[batch {i:02d}] starting {MOLS_PER_BATCH} mols ...', flush=True)

    wrapper = (
        f"import sys, runpy; sys.path.insert(0, {TARGETDIFF!r}); "
        f"sys.argv = ['sample_for_pocket.py', {CONFIG!r}, "
        f"'--pdb_path', {POCKET!r}, "
        f"'--num_samples', '{MOLS_PER_BATCH}', "
        f"'--batch_size', '{PER_GPU_BATCH}', "
        f"'--result_path', {out_dir!r}]; "
        f"runpy.run_path({SCRIPT!r}, run_name='__main__')"
    )
    env = os.environ.copy()
    env['PYTHONPATH'] = TARGETDIFF + os.pathsep + env.get('PYTHONPATH', '')

    t0 = time.time()
    r = subprocess.run([CONDA_PY, '-c', wrapper],
                       cwd=TARGETDIFF, capture_output=True, text=True, env=env)
    print(f'[batch {i:02d}] done in {(time.time()-t0)/60:.1f} min | rc={r.returncode}')
    if r.returncode != 0:
        print('STDERR (last 1500 chars):\n', r.stderr[-1500:])
        break

print(f'\n=== Loop finished in {(time.time()-t_all)/60:.1f} min ===')

done, total_sdfs = 0, 0
for i in range(N_BATCHES):
    out_dir = os.path.join(BATCHES, f'batch_{i:02d}')
    if batch_done(out_dir):
        done += 1
        sdf_dir = os.path.join(out_dir, 'sdf')
        if os.path.isdir(sdf_dir):
            total_sdfs += len([f for f in os.listdir(sdf_dir) if f.endswith('.sdf')])
print(f'Completed batches: {done}/{N_BATCHES}  |  SDFs on disk: {total_sdfs}')


## Merge all batches into a single `1M17_raw.sdf`

Concatenates every per-molecule SDF into `results/generated/1M17_raw.sdf` with
canonical SMILES attached as an SDF property, plus a validity report.


In [ ]:
# -- Cell 7: Merge per-batch SDFs into 1M17_raw.sdf --
# Runs in the conda Python 3.11 env (where rdkit lives) so the kernel's
# Python version doesn't matter.
import subprocess, os

CONDA_PY = '/usr/local/bin/python3.11'
merge_code = f"""
import os, glob
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

BATCHES = '{PROJECT_ROOT}/results/generated/1M17_batches'
OUT_SDF = '{PROJECT_ROOT}/results/generated/1M17_raw.sdf'
os.makedirs(os.path.dirname(OUT_SDF), exist_ok=True)

sdf_files = sorted(glob.glob(os.path.join(BATCHES, 'batch_*', 'sdf', '*.sdf')))
print(f'Found {{len(sdf_files)}} per-molecule SDFs')

writer = Chem.SDWriter(OUT_SDF)
n_valid, n_invalid, smi_set = 0, 0, set()
for sdf in sdf_files:
    try:
        for mol in Chem.SDMolSupplier(sdf, sanitize=True, removeHs=False):
            if mol is None:
                n_invalid += 1; continue
            smi = Chem.MolToSmiles(mol)
            if not smi:
                n_invalid += 1; continue
            mol.SetProp('_Name', os.path.basename(sdf).replace('.sdf', ''))
            mol.SetProp('SMILES', smi)
            writer.write(mol)
            smi_set.add(smi); n_valid += 1
    except Exception:
        n_invalid += 1
writer.close()

print(f'Valid          : {{n_valid}}')
print(f'Invalid        : {{n_invalid}}')
print(f'Unique SMILES  : {{len(smi_set)}}')
print(f'Written to     : {{OUT_SDF}}  ({{os.path.getsize(OUT_SDF)/1024:.1f}} KB)')
"""

r = subprocess.run([CONDA_PY, '-c', merge_code], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
